[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeiraDavidson/compute_module1/blob/main/notebooks/lab_05_dots_and_lines.ipynb)

# Lab 5: Dots and Lines

*From* **A First Taste of Algorithms — and the Machines That Learn** *(The Curious Family Library). A computational lab: run the cells in order. A few labs end with a deliberately broken “Break it” cell — that one is meant to fail, and seeing how is the point.*

*Accompanies Chapter 5. The chapter walked graphs three ways with coins and napkins; here the same three walks run on the machine. Two small tools arrive — each because the graph demands it — and a maze of your own design gets solved by ripple, by thread, and then, with one word changed, by each pretending to be the other.*

## 1. A graph in the machine

The chapter's school, verbatim — a dictionary with one entry per vertex, holding its list of neighbors:

In [ ]:
hallways = {
    "gym":       ["lobby"],
    "lobby":     ["gym", "library", "cafeteria"],
    "library":   ["lobby", "lab"],
    "cafeteria": ["lobby", "lab"],
    "lab":       ["library", "cafeteria"],
}

print(hallways["lobby"])

Notice the convention: every edge appears *twice* (gym lists lobby, lobby lists gym), because a hallway can be walked both ways. Check the dictionary against a drawing of the school — then deliberately desynchronize it (delete `"gym"` from lobby's list but keep lobby in gym's) and hold that thought for Break-it 1.

## 2. The ripple

BFS, with the queue kept honest in plain sight: the waiting line is a list, and a marker `head` points at who gets served next. Serving means moving the marker; joining means appending. Nothing is ever deleted — the line keeps its history, which will turn out to be a gift.

One new tool, arriving with its debt: the **set**. The visited-check must be fast and duplicate-proof; a set is a bag that holds each thing at most once and answers `in` instantly. Three moves: `seen = set()`, `seen.add(x)`, `x in seen`.

In [ ]:
def bfs(graph, start):
    queue = [start]
    head = 0
    seen = {start}                      # a set with one thing in it
    dist = {start: 0}                   # the notebook: vertex -> hops
    while head < len(queue):
        here = queue[head]
        head = head + 1                 # serve the front of the line
        for neighbor in graph[here]:
            if neighbor not in seen:
                seen.add(neighbor)
                dist[neighbor] = dist[here] + 1
                queue.append(neighbor)  # join the back of the line
    return dist

print(bfs(hallways, "gym"))

Run it and check every distance against your drawing. Then print `queue` itself after the run (add a print, or return it too): the line's history *is* the rings, in order — read it aloud as "ring 0, ring 1, ring 2" and confirm the bureaucracy did what the chapter promised.

## 3. A maze is a graph wearing a costume

Build a maze as a list of strings — `#` wall, `.` floor, `S` start, `E` exit:

In [ ]:
maze = [
    "#########",
    "#S..#...#",
    "#.#.#.#.#",
    "#.#...#.#",
    "#.###.#.#",
    "#.....#E#",
    "#########",
]

Where is the graph? Every floor cell is a vertex; two floor cells side-by-side share an edge. We never build the dictionary — the maze *answers neighbor queries on demand* (look up, down, left, right; a graph is anywhere that can answer "who are your neighbors?").

A cell's name is its (row, column) position — and here the second tool arrives with its debt. Positions must live in the `seen` set and the `dist` notebook, and Python's rule is that set members and notebook keys must be *unchangeable* things; a list, which can be appended to and edited, is not allowed. The unchangeable list is called a **tuple**: written `(5, 7)` with round brackets, indexed like a list, frozen forever at birth. Positions are tuples because positions must be lookupable.

In [ ]:
def maze_bfs(maze, start):
    queue = [start]
    head = 0
    seen = {start}
    dist = {start: 0}
    while head < len(queue):
        here = queue[head]
        head = head + 1
        r = here[0]
        c = here[1]
        for next_pos in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
            if maze[next_pos[0]][next_pos[1]] != "#":
                if next_pos not in seen:
                    seen.add(next_pos)
                    dist[next_pos] = dist[here] + 1
                    queue.append(next_pos)
    return dist

distances = maze_bfs(maze, (1, 1))      # S is at row 1, column 1
print(distances[(5, 7)])                # E is at row 5, column 7

The ripple's answer is the shortest path length in steps. Verify it by tracing the maze with a pencil — then design your *own* maze (keep the `#` border, it is what spares the code from falling off the edge) with at least two routes from S to E of different lengths, and confirm BFS reports the shorter.

## 4. The one word

The chapter claimed BFS and DFS differ by which end of the line you serve. Test the claim with the smallest possible edit — serve the *back*:

In [ ]:
def dfs(graph, start):
    line = [start]
    seen = {start}
    order = []
    while len(line) > 0:
        here = line.pop()               # .pop(): remove and return the LAST item
        order.append(here)
        for neighbor in graph[here]:
            if neighbor not in seen:
                seen.add(neighbor)
                line.append(neighbor)
    return order

print(dfs(hallways, "gym"))
print(bfs(hallways, "gym"))

(`.pop()` is the one new move: take the last item off a list. A line served from the back is called a **stack** — the call stack of Chapter 3 is one, which is why the chapter could say DFS *is* recursion.) Compare the two visit orders on the school, then on your maze (adapt as in §3): the ripple sweeps level by level; the thread plunges. Same dots, same lines, opposite temperaments, one word.

## 5. Price tags on the town

The chapter's town, with weights — a dictionary of neighbor lists where each neighbor comes with a price, as a `(place, minutes)` tuple:

In [ ]:
town = {
    "Home":    [("Park", 2), ("School", 9), ],
    "Park":    [("Home", 2), ("School", 5), ("Library", 9), ("Gym", 11)],
    "School":  [("Park", 5), ("Home", 9), ("Library", 1), ("Cafe", 7)],
    "Library": [("School", 1), ("Park", 9), ("Cafe", 4)],
    "Cafe":    [("Library", 4), ("School", 7), ("Gym", 3)],
    "Gym":     [("Park", 11), ("Cafe", 3)],
}

Dijkstra's protocol, exactly as you played it with coins — tags, settle the cheapest, relax its edges:

In [ ]:
def dijkstra(graph, start):
    tags = {start: 0}                   # tentative prices; absent = infinity
    settled = set()
    while len(settled) < len(graph):
        cheapest = None                 # scan for the cheapest unsettled tag
        for place in tags:
            if place not in settled:
                if cheapest is None or tags[place] < tags[cheapest]:
                    cheapest = place
        if cheapest is None:
            break                       # the rest is unreachable
        settled.add(cheapest)
        for edge in graph[cheapest]:
            place = edge[0]
            cost = tags[cheapest] + edge[1]
            if place not in tags or cost < tags[place]:
                tags[place] = cost      # relax: the detour wins
    return tags

print(dijkstra(town, "Home"))

(`None` is Python's word for "nothing yet" — the scan starts with no champion and crowns the first candidate.) Check every tag against the chapter's hand-run: Park 2, School 7, Library 8, Café 12, Gym 13 — five answers, five detours. Then rebuild `town` as **your actual neighborhood**, six or seven places with honest walking-minutes, and find something true you didn't know about your own streets.

## Break it

Predict first. Run second. Explain any gap third.

**1.** The desynchronized graph from §1: delete `"gym"` from lobby's neighbor list (keep lobby in gym's). Predict `bfs(hallways, "gym")` and then `bfs(hallways, "lab")` — same reachability or different? You have built a *one-way door*. Real graphs (the web, who-follows-whom) are full of them; what single sentence of the §1 convention did you break, and which of the two BFS runs is the one that can no longer find everything?

**2.** In `maze_bfs`, remove the border: change your maze's top row to open floor and run from S. Predict the failure before running — wrong answer, error, or infinite loop? — then read the error's last line and explain which innocent-looking expression walked off the world. (Two repairs exist: rebuild the wall, or teach the neighbor-check to refuse positions outside the maze. Implement the second; the first you already had.)

**3.** *The road that pays you.* First, make Dijkstra's promises audible: add one line right after `settled.add(cheapest)` — `print("settled:", cheapest, "at", tags[cheapest])` — and rerun the honest town. Compare the announcements with the final notebook: they agree perfectly, every time, on every honest map (that agreement *is* the deep-water theorem talking). Now build the chapter's Problem 3.3 graph using Break-it 1's one-way doors — `{"Home": [("B", 2), ("C", 3)], "B": [], "C": [("B", -2)]}` — predict the announcements, and run it. Catch the exact moment the algorithm publicly settles a lie (which vertex, announced at what price, versus the truth you can see by eye?). Then two follow-ups: (a) point to the clause of the deep-water proof that this graph kills, quoting it; (b) the *final* notebook actually shows the right number anyway — find the line of our code that quietly forgave the lie after the announcement, and explain why you still must not trust this on bigger graphs (what if the lying vertex's wrong tag had been used to relax *its* neighbors before the correction arrived?). One warning while experimenting: keep negative edges one-way here. A negative *two*-way road lets a walker pace back and forth earning money forever, and "cheapest route" stops meaning anything — the question itself, not just the algorithm, dies. (Restore the town when done; it has suffered enough.)

**4.** *The knight's ripple* (finishing Problem 2.5). A chessboard is rows and columns 0–7; a knight at (r, c) can jump to eight positions: (r±1, c±2) and (r±2, c±1), board permitting. Adapt `maze_bfs` — same skeleton, different neighbor list, and the "board permitting" check is Break-it 2's repair earning rent. Compute the fewest moves from (0, 0) to (7, 7), compare with the prediction you wrote in the problem set, and then print the full `dist` notebook and find the squares that are *hardest* for a corner knight to reach. Are they where you'd have guessed?



<div style="page-break-after: always;"></div>